# 02 — Stage 1 to tube-centric Stage 2 HDF5

This notebook verifies Stage 1, organizes every requested dataset by tube, creates preprocessing groups, and audits Stage 2 raw/preprocessed/LOT structures.

In [ ]:
from pathlib import Path
import h5py
import matplotlib.pyplot as plt
import seaborn as sns

from flowlot.io import Stage2Organizer, audit_stage1, audit_stage2

## Configuration

Use `strict` marker policy when every patient must have identical ordered markers; use `intersection` to retain the ordered common markers in each tube. Marker subsets may be global lists, per-tube dictionaries, or `None` for all available markers.

In [ ]:
STAGE1 = Path('../data/stage1_raw_data.h5')
STAGE2 = Path('../data/stage2_analytics.h5')
CONFIGURATIONS = [
    {
        'dataset': 'BLAST110', 'cells': 1000, 'marker_policy': 'intersection',
        'preprocess': 'common12', 'markers': None, 'arcsinh_cofactor': None,
    },
]
RUN_ORGANIZE = False  # Safety switch.

## Verify the input Stage 1 file

Do not organize a Stage 1 file with invariant failures. The inventory also exposes patient/tube coverage and marker dimensions before conversion.

In [ ]:
stage1_inventory, stage1_issues = audit_stage1(STAGE1)
display(stage1_inventory.groupby(['dataset', 'cell_count', 'tube', 'label']).agg(
    patients=('patient_id', 'nunique'), cells=('n_cells', 'sum'), markers=('n_markers', 'first')
).reset_index())
display(stage1_issues)
assert stage1_issues.empty, 'Correct Stage 1 errors before organizing Stage 2'

## Organize and preprocess

Each configuration replaces only its own `/{dataset}/{cells}` Stage 2 group. Other datasets remain intact. Preprocessing keeps one matrix per raw patient.

In [ ]:
if RUN_ORGANIZE:
    organizer = Stage2Organizer(STAGE1, STAGE2)
    for config in CONFIGURATIONS:
        organizer.organize(
            config['dataset'], config['cells'], marker_policy=config['marker_policy'], overwrite=True
        )
        organizer.add_preprocess(
            config['dataset'], config['cells'], config['preprocess'],
            marker_subset=config['markers'], arcsinh_cofactor=config['arcsinh_cofactor'],
        )
    print('Created', STAGE2.resolve())
else:
    print('Dry run. Set RUN_ORGANIZE=True after Stage 1 passes verification.')

## Verify Stage 2 and summarize every dataset

Checks cover metadata lengths, patient alignment, marker dimensions, preprocessing coverage, finite values, embedding rows, and flattened LOT widths.

In [ ]:
if STAGE2.exists():
    inventories, stage2_issues = audit_stage2(STAGE2)
    raw, processed, embeddings = inventories['raw'], inventories['preprocess'], inventories['embeddings']
    raw_summary = raw.groupby(['dataset', 'cell_count', 'tube', 'label']).agg(
        patients=('patient_id', 'nunique'), stored_cells=('n_cells', 'sum'),
        median_cells=('n_cells', 'median'), markers=('n_markers', 'first'),
        finite_fraction=('finite_fraction', 'min'),
    ).reset_index()
    display(raw_summary)
    display(processed.groupby(['dataset', 'cell_count', 'tube', 'preprocess']).agg(
        patients=('patient_id', 'nunique'), markers=('n_markers', 'first'),
        finite_fraction=('finite_fraction', 'min'),
    ).reset_index())
    display(embeddings)
    display(stage2_issues)
    assert stage2_issues.empty, 'Stage 2 verification failed; inspect stage2_issues'
else:
    print('Stage 2 does not exist yet.')

In [ ]:
if STAGE2.exists():
    coverage = raw.assign(present=1).pivot_table(
        index=['dataset', 'cell_count', 'patient_id', 'label'], columns='tube',
        values='present', aggfunc='max', fill_value=0,
    )
    display(coverage.head(20))
    for dataset, values in raw.groupby('dataset'):
        figure, axis = plt.subplots(figsize=(6, 3))
        sns.boxplot(data=values, x='tube', y='n_cells', hue='label', ax=axis)
        axis.set_title(f'{dataset}: stored cells per patient/tube')
        sns.despine()
        plt.show()

In [ ]:
if STAGE2.exists():
    with h5py.File(STAGE2) as handle:
        print('schema:', dict(handle.attrs))
        for dataset in handle:
            for cells in handle[dataset]:
                print(f'/{dataset}/{cells} tubes:', sorted(handle[f'{dataset}/{cells}']))